# 01 - Data pipeline (final, reproducible)
Run the cells **top to bottom** (Runtime -> Restart and run all). Takes ~2 minutes.

Needs these files in your Drive project folder:
- `career_utils.py`, `career_pipeline.py` (the two modules)
- `raw/indian-job-market-dataset-2025.xlsx`
- `processed/p_it_skills_v1.pkl` (frozen IT-detector scores)

Produces `processed/jobs_final.pkl` and `processed/skill_vocab_final.pkl`.

In [1]:
import os, sys
from pathlib import Path
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path("/content/drive/MyDrive/Job recommendation nd skill gap analysis")
except ImportError:                      # running outside Colab
    BASE = Path(os.environ["CAREER_BASE"])
sys.path.insert(0, str(BASE))
import pickle, numpy as np, pandas as pd
import career_pipeline as cp
from career_utils import OTHER, AMBIG
(BASE / "processed").mkdir(exist_ok=True)
print("project folder:", BASE)

Mounted at /content/drive
project folder: /content/drive/MyDrive/Job recommendation nd skill gap analysis


## 1. Load and clean the raw listings
HTML/URL/e-mail/phone removal, jobId de-duplication, exact-content de-duplication.

In [2]:
df = cp.load_and_clean(BASE / "raw" / "indian-job-market-dataset-2025.xlsx")

rows: 97,929 -> jobId dedupe 97,679 -> content dedupe 93,729


## 2. Family labels from title rules (final rule set)

In [3]:
df = cp.add_family(df)
print(df["family"].value_counts().to_string())

IT listings by title rules: 22914
family
Other / Non-IT or Unclear       70815
Software Developer               7909
ERP / CRM Consultant             4013
IT Services - Ambiguous Role     1513
Data Engineer                    1444
QA / Testing                     1256
Data Scientist / ML-AI           1093
DevOps / Cloud                   1047
IT Support / Sys & Network        955
Frontend / Web                    802
IT Architect                      515
Business / Systems Analyst        514
Cybersecurity                     500
Data / BI Analyst                 485
Tech Project / Product            380
Mobile Developer                  257
Database Admin                    231


## 3. Skills: tag cleaning, aliases, stop-words, rare tags, merges, IT vocabulary (>=10 IT listings)

In [4]:
df, skill_vocab = cp.add_skills(df)

IT skill vocabulary: 1951


## 4. IT detector scores (frozen)
The skills-only detector's out-of-fold scores were **saved** and used to stratify the human validation sample,
so the pipeline loads them instead of recomputing. (`cp.build_detector(df)` recomputes them; the result differs slightly
because scikit-learn versions assign GroupKFold folds differently - about 1% of listings move across a tier boundary.)

In [5]:
p_it = cp.load_frozen_detector(df, BASE / "processed" / "p_it_skills_v1.pkl")
print(p_it.describe().round(3).to_dict())

{'count': 93729.0, 'mean': 0.229, 'std': 0.327, 'min': 0.0, '25%': 0.005, '50%': 0.031, '75%': 0.408, 'max': 1.0}


## 5. Core / extended tiers and predicted families (prior correction alpha = 0.5)

In [6]:
ext = cp.build_extended(df, p_it)
print()
print(ext[ext["family_source"].isin(["rule", "predicted"])]["family_final"].value_counts().to_string())

it_tier
non_IT_or_unknown    60119
core_rule            21401
extended_mid          6235
extended_high         4461
core_ambiguous        1513

family_final
Software Developer            10295
ERP / CRM Consultant           5534
QA / Testing                   2313
IT Support / Sys & Network     1896
Data Engineer                  1888
DevOps / Cloud                 1715
Data Scientist / ML-AI         1592
Frontend / Web                 1170
Data / BI Analyst              1131
Business / Systems Analyst     1035
Tech Project / Product          958
Cybersecurity                   929
IT Architect                    894
Database Admin                  390
Mobile Developer                357


## 6. Sanity checks (the notebook stops here if anything drifted)

In [7]:
assert len(ext) == 93_729, len(ext)
assert int(ext["family"].ne(OTHER).sum()) == 22_914
assert len(skill_vocab) == 1_951
tiers = ext["it_tier"].value_counts()
assert tiers["core_rule"] == 21_401 and tiers["core_ambiguous"] == 1_513
assert tiers["extended_mid"] == 6_235 and tiers["extended_high"] == 4_461
print("all checks passed")

all checks passed


## 7. (Optional) compare with the earlier saved version

In [8]:
old_path = BASE / "processed" / "jobs_extended_v2.pkl"
if old_path.exists():
    old = pd.read_pickle(old_path)
    same = (ext["family_final"].values == old["family_final"].values)
    print(f"family_final agreement with jobs_extended_v2: {same.mean():.5f}  ({int((~same).sum())} rows differ)")
else:
    print("jobs_extended_v2.pkl not found - skipped")

family_final agreement with jobs_extended_v2: 1.00000  (0 rows differ)


## 8. Save the final artifacts

In [9]:
ext.to_pickle(BASE / "processed" / "jobs_final.pkl")
pickle.dump(skill_vocab, open(BASE / "processed" / "skill_vocab_final.pkl", "wb"))
print("saved jobs_final.pkl", ext.shape, "| skill_vocab_final.pkl", len(skill_vocab))

saved jobs_final.pkl (93729, 31) | skill_vocab_final.pkl 1951
